Libraries

In [1]:
import pandas as pd
import numpy as np
from google.colab import files

Dataset

In [2]:
places = pd.read_csv("/content/PLACES__Census_Tract_Data_(GIS_Friendly_Format),_2025_release_20261019.csv", dtype=str)
svi = pd.read_csv("/content/SVI_2022_US.csv", dtype=str)
diseases = ["DIABETES", "BPHIGH", "CASTHMA", "COPD", "CHD"]
# mapping boroughs for NYC case
boroughs = {"36005": "Bronx", "36047": "Brooklyn", "36061": "Manhattan", "36081": "Queens", "36085": "Staten Island"}



In [3]:
# county = first 5 digits of the tract code
places["CountyFIPS"] = places["TractFIPS"].str[:5]
# remove commas from population and make it a number
places["TotalPop18plus"] = pd.to_numeric(places["TotalPop18plus"].str.replace(",", ""))

# prevalence as a number, then estimated adults with each disease
for d in diseases:
    places[d + "_CrudePrev"] = pd.to_numeric(places[d + "_CrudePrev"])
    places[d + "_EstCount"] = places[d + "_CrudePrev"] / 100 * places["TotalPop18plus"]

# columns to keep
columns = ["TractFIPS", "StateAbbr", "CountyFIPS", "TotalPop18plus"]
for d in diseases:
    columns += [d + "_CrudePrev", d + "_EstCount"]

# Q5: NYC only, with borough names
q5 = places[places["CountyFIPS"].isin(boroughs)].copy()
q5["Borough"] = q5["CountyFIPS"].map(boroughs)
q5[columns + ["Borough"]].to_csv("q5.csv", index=False)

# SVI: -999 means missing
svi["RPL_THEMES"] = pd.to_numeric(svi["RPL_THEMES"])
svi["RPL_THEMES"] = svi["RPL_THEMES"].mask(svi["RPL_THEMES"] == -999)
# Q6: add SVI score by matching tract code
q6 = places.merge(svi[["FIPS", "RPL_THEMES"]], left_on="TractFIPS", right_on="FIPS", how="left")
q6[columns + ["RPL_THEMES"]].to_csv("q6.csv", index=False)

# download both files
files.download("q5.csv")
files.download("q6.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>